# External Validation — Roboflow "AI-Leprosy" (Stage-1 detector)
نوتبوك واحد لتقييم الـ Stage-1 ensemble على الـ external dataset (Roboflow `AI-Leprosy`) فقط، بنفس الكود والإعدادات المستخدمة في الورقة. **شغّليه Run all على Colab (GPU)**.

ما يعمله:
1. يعيد توليد تقسيم Stage-1 (seed 42) ويتأكد إنه يعيد أرقام الورقة (97.01%).
2. يجمع صور Roboflow (test + valid فقط، يستبعد الـ train المتضاعف بالـ augmentation، ويستبعد تعارض الـ labels).
3. مقاييس كاملة مع 95% bootstrap CI (accuracy, recall, specificity, balanced accuracy, precision, F1, AUC) + baseline "كله جُذام".
4. Reinhard color normalization بمرجعين.
5. Thresholds محسوبة من الـ in-distribution validation فقط (بدون tuning على الـ external).
6. Error analysis (FN/FP، توزيع الـ probabilities، حسب split وحسب upload batch).
7. Perceptual-hash (pHash/dHash): هل صور Roboflow near-duplicates لصور التدريب؟

كل قسم داخل `section(...)`: لو فشل قسم يطبع الخطأ ويكمّل.

**المتطلبات:** Stage-1 checkpoints + مجلد الداتا الأصلية (31 class) + مجلد `AI-Leprosy.v1i.coco` على Drive (المسارات معدّلة في الخلية الثالثة).

In [ ]:
!pip install -q imagehash

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, re, time, random, platform, traceback, warnings, hashlib, shutil
from collections import defaultdict, Counter
from contextlib import contextmanager
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 200)

T0 = time.time()
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ------------------------- PATHS (same as the old notebooks) -------------------------
PROJECT_DIR   = "/content/drive/MyDrive/skin_disease_project"
RAW_DATA_DIR  = "/content/drive/MyDrive/Atlas dan ISIC2019 (31 classes)"
ROBOFLOW_DIR  = "/content/drive/MyDrive/AI-Leprosy.v1i.coco"
STAGE1_CKPT = {
    "efficientnet_b3": f"{PROJECT_DIR}/outputs/checkpoints/efficientnet_b3_best.pt",
    "convnext_tiny":   f"{PROJECT_DIR}/outputs/checkpoints/convnext_tiny_best.pt",
    "resnet50":        f"{PROJECT_DIR}/outputs/checkpoints/resnet50_best.pt",
}
OUT_DIR = f"{PROJECT_DIR}/roboflow_external_validation"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAMES = ["efficientnet_b3", "convnext_tiny", "resnet50"]
IMAGE_SIZE, DROPOUT_RATE = 300, 0.4
VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp")
LEPROSY_FOLDERS = ["Leprosy Tuberculoid", "Leprosy Lepromatous", "Leprosy Borderline"]
SUBTYPES = LEPROSY_FOLDERS                      # Stage-2 class order (same as training)
SUBTYPE_TO_IDX = {n: i for i, n in enumerate(SUBTYPES)}
SPLIT_RATIOS = (0.70, 0.15, 0.15)
N_BOOT = 1000
NEAR_DUP_T = 6          # pHash Hamming distance <= this => near-duplicate
RUN_HASHING = True      # set False to skip the slow hashing sections

# numbers reported in the paper (used only to VERIFY reproduction)
REPORTED_S1 = {
    "efficientnet_b3": dict(acc=96.47, prec=87.88, rec=86.14, f1=87.00, auc=0.970),
    "convnext_tiny":   dict(acc=96.20, prec=83.49, rec=90.10, f1=86.67, auc=0.980),
    "resnet50":        dict(acc=96.20, prec=83.49, rec=90.10, f1=86.67, auc=0.974),
    "ensemble":        dict(acc=97.01, prec=87.62, rec=91.09, f1=89.32, auc=0.981),
}
RESULTS = {}            # collected for the final JSON summary

@contextmanager
def section(name):
    print("\n" + "=" * 90); print(name); print("=" * 90)
    t = time.time()
    try:
        yield
    except Exception:
        print("❌ SECTION FAILED:", name); traceback.print_exc()
    finally:
        print(f"[{name}] finished in {time.time()-t:.1f}s")

for k, p in [("PROJECT_DIR", PROJECT_DIR), ("RAW_DATA_DIR", RAW_DATA_DIR), ("ROBOFLOW_DIR", ROBOFLOW_DIR)] + [(f"S1:{m}", p) for m, p in STAGE1_CKPT.items()]:
    print(f"{'✅' if os.path.exists(p) else '❌ MISSING'}  {k}: {p}")

In [ ]:
with section("0) Environment"):
    import torch, torchvision, sklearn, scipy
    print("python      :", sys.version.split()[0])
    print("platform    :", platform.platform())
    print("torch       :", torch.__version__, "| torchvision:", torchvision.__version__)
    print("numpy/pandas:", np.__version__, pd.__version__, "| sklearn:", sklearn.__version__, "| scipy:", scipy.__version__)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device      :", DEVICE)
    if DEVICE.type == "cuda":
        print("GPU         :", torch.cuda.get_device_name(0))
        print("GPU memory  : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    else:
        print("⚠️ NO GPU — change runtime to GPU for speed.")
    RESULTS["env"] = dict(torch=torch.__version__, torchvision=torchvision.__version__,
                          gpu=(torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"))

In [ ]:
# ============================ UTILITIES (metrics / bootstrap / tests) ============================
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve

def counts(y, pred):
    y = np.asarray(y).astype(int); pred = np.asarray(pred).astype(int)
    return (int(((y == 1) & (pred == 1)).sum()), int(((y == 1) & (pred == 0)).sum()),
            int(((y == 0) & (pred == 1)).sum()), int(((y == 0) & (pred == 0)).sum()))   # tp, fn, fp, tn

def metrics_from_counts(tp, fn, fp, tn):
    n = tp + fn + fp + tn
    rec  = 100 * tp / (tp + fn) if tp + fn else float("nan")
    spec = 100 * tn / (tn + fp) if tn + fp else float("nan")
    prec = 100 * tp / (tp + fp) if tp + fp else 0.0
    f1   = 100 * 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    den  = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc  = (tp * tn - fp * fn) / den if den > 0 else 0.0
    return dict(acc=100 * (tp + tn) / n, prec=prec, rec=rec, spec=spec, bacc=(rec + spec) / 2, f1=f1, mcc=mcc)

def full_metrics(y, s, thr=0.5, with_auc=True):
    y = np.asarray(y).astype(int); s = np.asarray(s, dtype=float)
    pred = (s >= thr).astype(int)
    m = metrics_from_counts(*counts(y, pred))
    m["auc"] = float(roc_auc_score(y, s)) if (with_auc and len(np.unique(y)) == 2) else float("nan")
    return m

METRIC_KEYS = ["acc", "prec", "rec", "spec", "bacc", "f1", "mcc", "auc"]

def bootstrap_metrics(y, s, thr=0.5, n_boot=N_BOOT, seed=SEED, keys=METRIC_KEYS):
    rng = np.random.default_rng(seed); y = np.asarray(y).astype(int); s = np.asarray(s, dtype=float); n = len(y)
    res = {k: [] for k in keys}
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) < 2: continue
        m = full_metrics(y[i], s[i], thr, with_auc=("auc" in keys))
        for k in keys: res[k].append(m[k])
    return {k: (float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))) for k, v in res.items()}

def paired_bootstrap_diff(y, sA, sB, thr=0.5, n_boot=N_BOOT, seed=SEED, keys=("acc", "f1", "rec", "auc")):
    """metric(A) - metric(B) on the SAME resampled images."""
    rng = np.random.default_rng(seed); y = np.asarray(y).astype(int); sA = np.asarray(sA, float); sB = np.asarray(sB, float); n = len(y)
    d = {k: [] for k in keys}
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) < 2: continue
        a = full_metrics(y[i], sA[i], thr); b = full_metrics(y[i], sB[i], thr)
        for k in keys: d[k].append(a[k] - b[k])
    out = {}
    for k, v in d.items():
        v = np.array(v); out[k] = dict(mean=float(np.nanmean(v)), lo=float(np.nanpercentile(v, 2.5)),
                                       hi=float(np.nanpercentile(v, 97.5)), p_gt0=float(np.nanmean(v > 0)))
    return out

def mcnemar_exact(correct_a, correct_b):
    a = np.asarray(correct_a, bool); b = np.asarray(correct_b, bool)
    a_only = int((a & ~b).sum()); b_only = int((~a & b).sum()); nd = a_only + b_only
    p = 1.0 if nd == 0 else float(stats.binomtest(min(a_only, b_only), nd, 0.5).pvalue)
    return a_only, b_only, p

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (100 * (c - h) / d, 100 * (c + h) / d)

def fmt_ci(m, ci, k, nd=1):
    return f"{m[k]:.{nd}f} [{ci[k][0]:.{nd}f}, {ci[k][1]:.{nd}f}]"

def youden_threshold(y, s):
    fpr, tpr, thr = roc_curve(y, s); j = tpr - fpr; j[~np.isfinite(thr)] = -9
    return float(thr[int(np.argmax(j))])

def sens_threshold(y, s, target=0.95):
    fpr, tpr, thr = roc_curve(y, s); ok = np.where((tpr >= target) & np.isfinite(thr))[0]
    return float(thr[ok[0]]) if len(ok) else float("nan")     # highest threshold that still reaches the target sensitivity

def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if (isinstance(o, float) and np.isnan(o)) else float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.ndarray): return to_py(o.tolist())
    return o
print("utilities ready ✅")

In [ ]:
# ============================ MODELS + INFERENCE HELPERS ============================
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

def build_stage1_model(arch):
    if arch == "efficientnet_b3":
        m = models.efficientnet_b3(weights=None); n = m.classifier[1].in_features
        m.classifier = nn.Sequential(nn.Dropout(p=DROPOUT_RATE), nn.Linear(n, 2))
    elif arch == "convnext_tiny":
        m = models.convnext_tiny(weights=None); n = m.classifier[2].in_features
        m.classifier[2] = nn.Sequential(nn.Dropout(p=DROPOUT_RATE), nn.Linear(n, 2))
    elif arch == "resnet50":
        m = models.resnet50(weights=None); n = m.fc.in_features
        m.fc = nn.Sequential(nn.Dropout(p=DROPOUT_RATE), nn.Linear(n, 2))
    else: raise ValueError(arch)
    return m

def build_stage2_model():
    m = models.efficientnet_b3(weights=None); n = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(p=DROPOUT_RATE), nn.Linear(n, 3))
    return m

stage1_models = {}
for name, path in STAGE1_CKPT.items():
    m = build_stage1_model(name); m.load_state_dict(torch.load(path, map_location=DEVICE)); m.to(DEVICE).eval()
    stage1_models[name] = m
print("loaded:", list(stage1_models))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EVAL_TF = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, paths, pre=None): self.paths, self.pre = list(paths), pre
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.pre is not None: img = self.pre(img)
        return EVAL_TF(img)

@torch.no_grad()
def predict_stage1(paths, pre=None, bs=32):
    """returns (n, 3) matrix of P(leprosy) per model, in MODEL_NAMES order. class index 0 == leprosy."""
    dl = DataLoader(ImgDS(paths, pre), batch_size=bs, shuffle=False, num_workers=2)
    outs = []
    for x in dl:
        x = x.to(DEVICE)
        outs.append(np.stack([F.softmax(stage1_models[n](x), 1)[:, 0].cpu().numpy() for n in MODEL_NAMES], axis=1))
    return np.concatenate(outs, 0)



In [ ]:
with section("1) Re-generate the Stage-1 split (same code/seed as training) and verify counts"):
    from sklearn.model_selection import train_test_split
    def collect_images_by_label():
        images = defaultdict(list)
        for root, dirs, files in os.walk(RAW_DATA_DIR):
            imgs = [f for f in files if f.lower().endswith(VALID_EXTS)]
            if not imgs: continue
            cn = os.path.basename(root)
            lab = "leprosy" if cn in LEPROSY_FOLDERS else "non_leprosy"
            for f in imgs: images[lab].append(os.path.join(root, f))
        return images

    def stratified_split(paths, ratios, seed):
        tr, va, te = ratios
        train_p, temp_p = train_test_split(paths, train_size=tr, random_state=seed)
        val_p, test_p = train_test_split(temp_p, train_size=va / (va + te), random_state=seed)
        return train_p, val_p, test_p

    imgs_by_label = collect_images_by_label()
    rows = []
    for lab, paths in imgs_by_label.items():
        tr, va, te = stratified_split(paths, SPLIT_RATIOS, SEED)
        for sp, pp in [("train", tr), ("val", va), ("test", te)]:
            for p in pp: rows.append(dict(path=p, label=lab, split=sp, cls=os.path.basename(os.path.dirname(p))))
    split_df = pd.DataFrame(rows); split_df["y"] = (split_df.label == "leprosy").astype(int)
    split_df.to_csv(f"{OUT_DIR}/stage1_split_regenerated.csv", index=False)
    ct = pd.crosstab(split_df.split, split_df.label).reindex(["train", "val", "test"]); ct["Total"] = ct.sum(1)
    print(ct)
    print("EXPECTED  (paper): train 470/2965, val 101/636, test 101/636")
    ok = (ct.loc["train", "leprosy"] == 470 and ct.loc["val", "leprosy"] == 101 and ct.loc["test", "leprosy"] == 101
          and ct.loc["train", "non_leprosy"] == 2965 and ct.loc["val", "non_leprosy"] == 636 and ct.loc["test", "non_leprosy"] == 636)
    print("split counts match:", "✅" if ok else "❌ (check)")
    print("\nnon-leprosy conditions in the pool (expected 28):", split_df[split_df.label == "non_leprosy"].cls.nunique())
    RESULTS["split_counts"] = ct.to_dict()

In [ ]:
with section("2) Stage-1 predictions on val/test (saved) + verify we reproduce the paper's numbers"):
    val_df  = split_df[split_df.split == "val"].reset_index(drop=True)
    test_df = split_df[split_df.split == "test"].reset_index(drop=True)
    P_val  = predict_stage1(val_df.path);  P_test = predict_stage1(test_df.path)
    y_val, y_test = val_df.y.values, test_df.y.values
    S_test = {m: P_test[:, i] for i, m in enumerate(MODEL_NAMES)}; S_test["ensemble"] = P_test.mean(1)
    S_val  = {m: P_val[:, i] for i, m in enumerate(MODEL_NAMES)};  S_val["ensemble"] = P_val.mean(1)
    np.savez(f"{OUT_DIR}/stage1_probs_val_test.npz", P_val=P_val, y_val=y_val, P_test=P_test, y_test=y_test,
             val_paths=val_df.path.values, test_paths=test_df.path.values)
    pd.DataFrame({"path": test_df.path, "y": y_test, **{f"p_{k}": v for k, v in S_test.items()}}).to_csv(f"{OUT_DIR}/stage1_test_probs.csv", index=False)

    rows, ok_all = [], True
    for k in MODEL_NAMES + ["ensemble"]:
        m = full_metrics(y_test, S_test[k]); ci = bootstrap_metrics(y_test, S_test[k])
        rep = REPORTED_S1[k]
        match = all(abs(m[x] - rep[x]) < (0.011 if x != "auc" else 0.0011) for x in ["acc", "prec", "rec", "f1", "auc"])
        ok_all &= match
        tp, fn, fp, tn = counts(y_test, (S_test[k] >= 0.5).astype(int))
        rows.append(dict(model=k, TP=tp, FN=fn, FP=fp, TN=tn, acc=fmt_ci(m, ci, "acc"), prec=fmt_ci(m, ci, "prec"),
                         rec=fmt_ci(m, ci, "rec"), spec=fmt_ci(m, ci, "spec"), f1=fmt_ci(m, ci, "f1"),
                         auc=fmt_ci(m, ci, "auc", 3), reproduces_paper="✅" if match else "❌"))
        RESULTS.setdefault("stage1_test", {})[k] = dict(metrics=m, ci=ci, counts=[tp, fn, fp, tn])
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nREPRODUCTION:", "✅ all numbers match the paper" if ok_all else
          "❌ some numbers differ -> the regenerated split is NOT identical to the one used in training (os.walk order?). Tell me.")
    RESULTS["stage1_reproduced"] = bool(ok_all)

In [ ]:
with section("6) Collect the external source (Roboflow 'AI-Leprosy')"):
    # ------------------ Roboflow (same logic as external-validation v3) ------------------
    def find_coco_json(split_dir, class_dir):
        for c in (os.path.join(split_dir, "_annotations.coco.json"), os.path.join(class_dir, "_annotations.coco.json")):
            if os.path.exists(c): return c
    def load_coco_labels(jp):
        coco = json.load(open(jp)); cat = {c["id"]: c["name"] for c in coco.get("categories", [])}
        im = {i["id"]: i["file_name"] for i in coco.get("images", [])}; out = {}
        for a in coco.get("annotations", []): out[os.path.basename(im.get(a["image_id"], ""))] = cat.get(a["category_id"], "unknown")
        return out
    rf, rf_excl = [], []
    for split in ["test", "valid"]:                      # 'train' excluded: Roboflow augmented copies
        sd = os.path.join(ROBOFLOW_DIR, split)
        for folder, label in [("leprosy", "leprosy"), ("non_leprosy", "non_leprosy")]:
            cd = os.path.join(sd, folder)
            if not os.path.exists(cd): print("missing", cd); continue
            jp = find_coco_json(sd, cd); cl = load_coco_labels(jp) if jp else {}
            for f in os.listdir(cd):
                if not f.lower().endswith(VALID_EXTS): continue
                fp = os.path.join(cd, f); c = cl.get(f)
                if c is not None:
                    norm = "leprosy" if ("not" not in c.lower() and "non" not in c.lower()) else "non_leprosy"
                    if norm != label: rf_excl.append(dict(path=fp, folder_label=label, coco_label=c)); continue
                rf.append(dict(path=fp, filename=f, label=label, split=split))
    RB = re.compile(r"_(jpg|jpeg|png|bmp)\.rf\.[0-9a-fA-F]+", re.I)
    seen_ids, rf_dedup = set(), []
    for s in rf:
        key = (RB.sub("", os.path.splitext(s["filename"])[0]).lower(), s["label"])
        if key in seen_ids: continue
        seen_ids.add(key); rf_dedup.append(s)
    print(f"Roboflow: collected {len(rf)}, conflicts excluded {len(rf_excl)}, after name-dedup {len(rf_dedup)}")
    pd.DataFrame(rf_excl).to_csv(f"{OUT_DIR}/roboflow_excluded_label_conflicts.csv", index=False)

    ext_base = {}
    for nm, lst in [("Roboflow", rf_dedup)]:
        d = pd.DataFrame(lst); d["y"] = (d.label == "leprosy").astype(int); ext_base[nm] = d
        print(f"  {nm}: n={len(d)} | leprosy={int(d.y.sum())} ({100*d.y.mean():.1f}%) | non-leprosy={int((1-d.y).sum())}")
    print("expected: Roboflow 120 (90 leprosy / 30 non-leprosy)")

In [ ]:
with section("7) External inference: original + Reinhard colour-normalisation (2 reference choices)"):
    import cv2
    def lab_stats(paths):
        ms, ss = [], []
        for fp in paths:
            im = cv2.imread(fp)
            if im is None: continue
            lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB).astype(np.float32).reshape(-1, 3)
            ms.append(lab.mean(0)); ss.append(lab.std(0))
        return np.mean(ms, 0), np.mean(ss, 0), len(ms)
    def make_reinhard(tm, ts, eps=1e-6):
        def f(pil):
            im = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR); lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB).astype(np.float32)
            sm, ss = lab.reshape(-1, 3).mean(0), lab.reshape(-1, 3).std(0)
            lab = np.clip((lab - sm) / (ss + eps) * ts + tm, 0, 255).astype(np.uint8)
            return Image.fromarray(cv2.cvtColor(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB))
        return f
    tr = split_df[split_df.split == "train"]; rng = np.random.default_rng(SEED)
    lp_ = tr[tr.label == "leprosy"].path.values
    lep_ref = list(rng.choice(lp_, min(150, len(lp_)), replace=False))
    mix_ref = list(rng.choice(tr.path.values, min(300, len(tr)), replace=False))
    m1, s1, n1 = lab_stats(lep_ref); m2, s2, n2 = lab_stats(mix_ref)
    print(f"reference A (leprosy-only, n={n1}): mean {np.round(m1,1)} std {np.round(s1,1)}")
    print(f"reference B (natural train mix, n={n2}): mean {np.round(m2,1)} std {np.round(s2,1)}")
    VARIANTS = {"original": None, "reinhard_lep_ref": make_reinhard(m1, s1), "reinhard_mix_ref": make_reinhard(m2, s2)}

    EXT = {}      # EXT[(source, variant)] = DataFrame with per-image probabilities
    for src, base in ext_base.items():
        for vn, pre in VARIANTS.items():
            P = predict_stage1(base.path, pre=pre); d = base.copy()
            for i, mname in enumerate(MODEL_NAMES): d[f"p_{mname}"] = P[:, i]
            d["p_ens"] = P.mean(1); EXT[(src, vn)] = d
            d.drop(columns=["path"]).to_csv(f"{OUT_DIR}/external_probs_{src}_{vn}.csv", index=False)
    print("inference done for", list(EXT))

    rows, pt = [], []
    ind = full_metrics(y_test, S_test["ensemble"]); ind_ci = bootstrap_metrics(y_test, S_test["ensemble"])
    for (src, vn), d in EXT.items():
        m = full_metrics(d.y, d.p_ens); ci = bootstrap_metrics(d.y, d.p_ens); tp, fn, fp, tn = counts(d.y, (d.p_ens >= .5).astype(int))
        pr_ = d.y.mean()
        rows.append(dict(source=src, variant=vn, n=len(d), prev_lep=f"{100*pr_:.0f}%", TP=tp, FN=fn, FP=fp, TN=tn,
                         acc=fmt_ci(m, ci, "acc"), recall=fmt_ci(m, ci, "rec"), specificity=fmt_ci(m, ci, "spec"),
                         bal_acc=fmt_ci(m, ci, "bacc"), prec=fmt_ci(m, ci, "prec"), f1=fmt_ci(m, ci, "f1"),
                         auc=fmt_ci(m, ci, "auc", 3), allpos_F1=f"{100*2*pr_/(1+pr_):.1f}"))
        RESULTS.setdefault("external", {})[f"{src}|{vn}"] = dict(metrics=m, ci=ci, counts=[tp, fn, fp, tn],
                                                                 recall_wilson=wilson(tp, tp + fn), spec_wilson=wilson(tn, tn + fp))
    print(f"(reference) in-distribution ensemble: acc {fmt_ci(ind, ind_ci, 'acc')} | recall {fmt_ci(ind, ind_ci, 'rec')} | spec {fmt_ci(ind, ind_ci, 'spec')} | auc {fmt_ci(ind, ind_ci, 'auc', 3)}\n")
    print(pd.DataFrame(rows).T.to_string(header=False))
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/external_metrics_table.csv", index=False)

    print("\nEffect of Reinhard vs original (paired, per image) — how many images actually flip?")
    for src in ext_base:
        o = EXT[(src, "original")]
        for vn in ["reinhard_lep_ref", "reinhard_mix_ref"]:
            v = EXT[(src, vn)]; co = ((o.p_ens >= .5).astype(int) == o.y).values; cv_ = ((v.p_ens >= .5).astype(int) == v.y).values
            a, b, p = mcnemar_exact(cv_, co); pred_changed = int(((o.p_ens >= .5) != (v.p_ens >= .5)).sum())
            print(f"  {src:9s} {vn:18s}: predictions changed on {pred_changed} imgs | right-after-only {a}, right-before-only {b}, McNemar p={p:.3f}"
                  f" | AUC {full_metrics(o.y,o.p_ens)['auc']:.3f} -> {full_metrics(v.y,v.p_ens)['auc']:.3f}")

In [ ]:
with section("8) Threshold analysis WITHOUT tuning on the evaluation data"):
    d = EXT[("Roboflow", "original")]
    thr_val_youden = youden_threshold(y_val, S_val["ensemble"]); thr_val_s95 = sens_threshold(y_val, S_val["ensemble"], 0.95)
    thr_rf = youden_threshold(d.y, d.p_ens)
    print(f"thresholds -> in-dist VAL Youden: {thr_val_youden:.3f} | in-dist VAL @95% sens: {thr_val_s95:.3f} | Roboflow self-tuned Youden: {thr_rf:.3f}")
    plans = [("default 0.50", .5), ("in-dist VAL Youden (clean)", thr_val_youden), ("in-dist VAL @95% sens (clean)", thr_val_s95),
             ("self-tuned Youden on Roboflow (OPTIMISTIC, exploratory)", thr_rf), ("0.05 (F1-optimal on Roboflow, exploratory)", .05)]
    rows = []
    for pn, t in plans:
        if not np.isfinite(t): continue
        m = full_metrics(d.y, d.p_ens, t, with_auc=False); tp, fn, fp, tn = counts(d.y, (d.p_ens >= t).astype(int))
        rows.append(dict(rule=pn, thr=round(t, 3), TP=tp, FN=fn, FP=fp, TN=tn, recall=round(m["rec"], 1), spec=round(m["spec"], 1),
                         bal_acc=round(m["bacc"], 1), prec=round(m["prec"], 1), f1=round(m["f1"], 1)))
    thr_tab = pd.DataFrame(rows); print(thr_tab.to_string(index=False)); thr_tab.to_csv(f"{OUT_DIR}/threshold_analysis.csv", index=False)
    prev = d.y.mean(); print(f"\nall-positive baseline F1 (label everything leprosy): {100*2*prev/(1+prev):.1f}")
    print("\nFull sweep (descriptive):")
    r = []
    for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
        m = full_metrics(d.y, d.p_ens, t, with_auc=False); r.append(dict(thr=t, recall=round(m["rec"], 1), spec=round(m["spec"], 1), bal_acc=round(m["bacc"], 1), f1=round(m["f1"], 1)))
    print(pd.DataFrame(r).to_string(index=False))
    RESULTS["thresholds"] = dict(val_youden=thr_val_youden, val_s95=thr_val_s95, rf_youden=thr_rf)

In [ ]:
with section("9) Error analysis (original images, threshold 0.5)"):
    for src in ["Roboflow"]:
        d = EXT[(src, "original")].copy(); d["pred"] = (d.p_ens >= .5).astype(int)
        fn = d[(d.y == 1) & (d.pred == 0)].sort_values("p_ens", ascending=False); fp = d[(d.y == 0) & (d.pred == 1)].sort_values("p_ens", ascending=False)
        bins = dict(near_0p35_0p5=int(((fn.p_ens >= .35) & (fn.p_ens < .5)).sum()), mid_0p2_0p35=int(((fn.p_ens >= .2) & (fn.p_ens < .35)).sum()),
                    confident_lt_0p2=int((fn.p_ens < .2).sum()), very_confident_lt_0p05=int((fn.p_ens < .05).sum()))
        print(f"\n== {src}: {len(fn)} false negatives (of {int(d.y.sum())} leprosy), {len(fp)} false positives (of {int((1-d.y).sum())} non-leprosy)")
        print("   FN by P(leprosy):", bins)
        cols = ["filename", "p_ens", "p_efficientnet_b3", "p_convnext_tiny", "p_resnet50"]
        print("   -- false negatives:"); print(fn[cols].round(3).to_string(index=False))
        print("   -- false positives:"); print(fp[cols].round(3).to_string(index=False))
        fn[cols].to_csv(f"{OUT_DIR}/errors_FN_{src}.csv", index=False); fp[cols].to_csv(f"{OUT_DIR}/errors_FP_{src}.csv", index=False)
        RESULTS.setdefault("error_bins", {})[src] = dict(n_fn=len(fn), n_fp=len(fp), **bins)
        print("   per-model recall on the leprosy images:",
              {m: round(float(100 * (d[d.y == 1][f"p_{m}"] >= .5).mean()), 1) for m in MODEL_NAMES})
        # recall by Roboflow split and by upload batch (file-name prefix) -> is the miss rate uniform?
        lep = d[d.y == 1].copy(); lep["batch"] = lep.filename.str.extract(r"^(.*?)_\d+_(?:jpg|jpeg|png)", expand=False).fillna("other")
        print("\n   recall by dataset split:", lep.groupby("split").apply(lambda g: round(100 * (g.pred == 1).mean(), 1)).to_dict(), "| n:", lep.split.value_counts().to_dict())
        print("   recall by upload batch :", lep.groupby("batch").apply(lambda g: (len(g), round(100 * (g.pred == 1).mean(), 1))).to_dict())


In [ ]:
with section("11) Perceptual-hash (pHash+dHash): are the Roboflow images near-duplicates of the Atlas/ISIC training pool?"):
    if not RUN_HASHING:
        print("RUN_HASHING=False -> skipped")
    else:
        import imagehash
        from concurrent.futures import ThreadPoolExecutor
        all_paths = list(dict.fromkeys(list(split_df.path) + [p for d in ext_base.values() for p in d.path]))
        cache_p = f"{OUT_DIR}/phash_cache.npz"; HC = {}
        if os.path.exists(cache_p):
            z = np.load(cache_p, allow_pickle=True); HC = {p: (a, b) for p, a, b in zip(z["paths"], z["ph"], z["dh"])}
        def hash_one(p):
            try:
                im = Image.open(p); im.draft("RGB", (256, 256)); im = im.convert("RGB"); im.thumbnail((256, 256))
                return p, np.array(imagehash.phash(im).hash, dtype=bool).ravel(), np.array(imagehash.dhash(im).hash, dtype=bool).ravel()
            except Exception as e:
                return p, None, None
        todo = [p for p in all_paths if p not in HC]; print(f"images: {len(all_paths)} | cached: {len(all_paths)-len(todo)} | to hash: {len(todo)}")
        with ThreadPoolExecutor(8) as ex:
            for i, (p, a, b) in enumerate(ex.map(hash_one, todo)):
                if a is not None: HC[p] = (a, b)
                if (i + 1) % 1000 == 0: print(f"  hashed {i+1}/{len(todo)}")
        ps = [p for p in all_paths if p in HC]
        np.savez(cache_p, paths=np.array(ps, dtype=object), ph=np.stack([HC[p][0] for p in ps]), dh=np.stack([HC[p][1] for p in ps]))
        failed = [p for p in all_paths if p not in HC]; print("hash failures:", len(failed))
        PH = lambda paths: np.stack([HC[p][0] for p in paths]); DH = lambda paths: np.stack([HC[p][1] for p in paths])

        def min_ham(A, B, chunk=256):
            Af, Bf = A.astype(np.float32), B.astype(np.float32); dmin = np.empty(len(A), int); amin = np.empty(len(A), int)
            bs = Bf.sum(1)[None, :]
            for s in range(0, len(A), chunk):
                a = Af[s:s + chunk]; d = a.sum(1)[:, None] + bs - 2 * (a @ Bf.T)
                amin[s:s + chunk] = d.argmin(1); dmin[s:s + chunk] = d.min(1).round().astype(int)
            return dmin, amin
        def bucket(d): return {f"<= {t}": int((d <= t).sum()) for t in (0, 2, 4, 6, 8, 10)}

        # (a) external vs the WHOLE Atlas/ISIC pool used by the project
        pool_paths = [p for p in split_df.path if p in HC]; pool_ph = PH(pool_paths)
        pool_meta = split_df.set_index("path")
        for src, base in ext_base.items():
            ps_ = [p for p in base.path if p in HC]; d, a = min_ham(PH(ps_), pool_ph)
            near = pd.DataFrame({"external": [os.path.basename(p) for p in ps_], "label": base.set_index("path").loc[ps_, "label"].values, "phash_dist": d,
                                 "nearest_train_img": [os.path.basename(pool_paths[i]) for i in a],
                                 "nearest_split": [pool_meta.loc[pool_paths[i], "split"] for i in a], "nearest_class": [pool_meta.loc[pool_paths[i], "cls"] for i in a],
                                 "dhash_dist": [int((HC[ps_[k]][1] != HC[pool_paths[a[k]]][1]).sum()) for k in range(len(ps_))]})
            near.sort_values("phash_dist").to_csv(f"{OUT_DIR}/leakage_external_{src}_vs_pool.csv", index=False)
            print(f"\n[{src}] nearest-neighbour pHash distance to ANY Atlas/ISIC image (n={len(ps_)}): median {np.median(d):.0f}, min {d.min()}")
            print("   counts:", bucket(d)); 
            fl = near[near.phash_dist <= NEAR_DUP_T]
            print(f"   near-duplicates (<= {NEAR_DUP_T}): {len(fl)}" + ("" if fl.empty else "\n" + fl.head(15).to_string(index=False)))
            RESULTS.setdefault("leakage_external", {})[src] = dict(n=len(ps_), min=int(d.min()), median=float(np.median(d)), buckets=bucket(d))


In [ ]:
with section("12) Final summary + zip"):
    RESULTS["elapsed_min"] = round((time.time() - T0) / 60, 1)
    json.dump(to_py(RESULTS), open(f"{OUT_DIR}/final_summary.json", "w"), indent=1)
    zp = shutil.make_archive(f"{PROJECT_DIR}/roboflow_external_validation", "zip", OUT_DIR)
    print("saved:", zp)
    for f in sorted(os.listdir(OUT_DIR)): print(f"  {f}  ({os.path.getsize(os.path.join(OUT_DIR, f))/1024:.0f} KB)")
    print(f"\nTotal run time: {RESULTS['elapsed_min']} min")
    print("\n✅ DONE. Now: File -> Download -> Download .ipynb (with outputs) and send it.")